In [ ]:
import geopandas as gpd
import pandas as pd

basline_blocks = gpd.read_file('data/traning_data/gatchina_baseline.geojson')
basline_blocks.head()


,residential,business,recreation,industrial,transport,special,agriculture,land_use,share,footprint_area,...,population,site_area,fsi,gsi,mxi,l,morphotype,area_accessibility,id,geometry
0,0.690207,0.211718,0.055976,0.0,0.042093,0.0,0.0,LandUse.RESIDENTIAL,0.690207,7383.849525,...,30.0,836277.821580,0.012109,0.008829,0.091908,1.371459,individual residential,10.063987,0,"POLYGON ((30.09367 59.55207, 30.09375 59.55215..."
1,0.970519,0.029484,0.000000,0.0,0.000000,0.0,0.0,LandUse.RESIDENTIAL,0.970519,15304.958732,...,2000.0,420144.041613,0.212156,0.036428,0.712492,5.824002,mid-rise,10.210398,1,"POLYGON ((30.09448 59.56164, 30.09448 59.56164..."
2,0.936589,0.010562,0.052849,0.0,0.000000,0.0,0.0,LandUse.RESIDENTIAL,0.936589,17141.628200,...,2275.0,302985.560227,0.303551,0.056576,0.786029,5.365401,mid-rise,8.669212,2,"POLYGON ((30.12546 59.58002, 30.12541 59.57998..."
3,0.121353,0.819171,0.059489,0.0,0.000000,0.0,0.0,LandUse.BUSINESS,0.819171,9453.530183,...,134.0,124546.982903,0.173716,0.075903,0.198114,2.288643,low-rise model,7.467434,3,"POLYGON ((30.12501 59.56588, 30.12497 59.56582..."
4,0.068737,0.931312,0.000000,0.0,0.000000,0.0,0.0,LandUse.BUSINESS,0.931312,8567.309559,...,56.0,119574.990985,0.156696,0.071648,0.095598,2.187029,low-rise model,8.285434,4,"POLYGON ((30.1249 59.56574, 30.12497 59.56582,..."


In [31]:
from fp2mp_baselines import GeneratorCriticBaseline, SingleAgentBaseline, CotBaseline


In [37]:
from langchain_ollama import ChatOllama

llm = ChatOllama(

    model="llama3",

    base_url="http://localhost:11434",

    temperature=0.5

)

print(llm.invoke("Скажи одним предложением, что такое город"))
task = "Город Санкт-Петербург. Разработай план уплотнения городского центра."

content='Город - это крупное населённое место, обладающее определенной территорией, инфраструктурой и социально-экономической деятельностью, в котором сосредоточены люди, организации и учреждения, которые взаимодействуют друг с другом и образуют уникальную культурно-историческую среду.' additional_kwargs={} response_metadata={'model': 'llama3', 'created_at': '2026-06-17T00:02:46.948359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4062126958, 'load_duration': 55467708, 'prompt_eval_count': 20, 'prompt_eval_duration': 380790708, 'eval_count': 78, 'eval_duration': 3605376625, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'} id='lc_run--019ed2e3-2844-7fb2-af41-1dd70d4c87b6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 20, 'output_tokens': 78, 'total_tokens': 98}


In [38]:
baseline = GeneratorCriticBaseline(generator_llm=llm)
state = baseline.invoke_state(task)

In [39]:
from typing import Any
def get_evaluation_for_block(block_id,goals:str,baseliine: Any):
    block = basline_blocks[basline_blocks['id'] == block_id]
    prompt = (
            "Ты градостроительный эксперт. Оцени, насколько сценарий соответствует целям "
            "социально-экономического развития.\n"
            "Верни только JSON без markdown в формате:\n"
            '{"score": 0.0, "reasoning": "краткое объяснение на русском"}\n\n'
            "Правила:\n"
            "- 0.0 означает, что сценарий противоречит целям.\n"
            "- 0.5 означает частичное соответствие.\n"
            "- 1.0 означает максимально сильное соответствие.\n"
            "- Используй только данные из сценария.\n"
            "- В reasoning укажи 1-3 причины оценки.\n\n"
            f"Цели СЭР:\n{goals}\n\n"
            "Сценарий:\n"
            f"{block}\n"
        )
    if block.empty:
        return None
    else:
        return baseline.invoke_state(prompt)

In [40]:
# Оценка бейзлайнов для случайных блоков
import random
random_block_id = random.choice(basline_blocks['id'].tolist())
state_1 = get_evaluation_for_block(random_block_id, "Развитие общественного транспорта, увеличение зеленых зон, поддержка малого бизнеса", baseline)

In [41]:
print(state_1["draft"])
print(state_1["critique"])
print(state_1["output"])

Here is a draft answer:

{"score": 0.5, "reasoning": "Сценарий частично соответствует целям СЭР. Общественный транспорт не развивается (0.0), что противоречит цели. Зеленые зоны присутствуют в сценарии (LandUse.RECREATION 1.0), но это не является основной функцией территории. Поддержка малого бизнеса также не очевидна. Однако, сценарий имеет рекреационную функцию, что может быть частично соответствующим целям."}
Review of the draft answer:

**Strengths:**

* The answer provides a clear and concise explanation for the score.
* It correctly identifies the lack of development in public transportation (0.0) as a major issue that contradicts the goal of social-economic development.

**Weaknesses/Omissions:**

* The reasoning does not explicitly state how the scenario partially corresponds to the goals of social-economic development, making it unclear what aspects of the scenario align with the goals.
* The answer does not mention the increase in green zones (LandUse.RECREATION 1.0) as a pos

In [42]:
baseline = SingleAgentBaseline(llm=llm)   
state_2 = get_evaluation_for_block(random_block_id, "Развитие общественного транспорта, увеличение зеленых зон, поддержка малого бизнеса", baseline)

In [ ]:
print(state_2["output"])

{"score": 0.5, "reasoning": "Сценарий частично соответствует целям социально-экономического развития, так как в нем есть зона рекреации (1.0), что может способствовать развитию зеленых зон и поддержке общественного транспорта. Однако, сценарий не учитывает увеличение зеленых зон и развитие общественного транспорта напрямую."}


In [44]:
baseline = CotBaseline(llm=llm)
state_3 = get_evaluation_for_block(random_block_id, "Развитие общественного транспорта, увеличение зеленых зон, поддержка малого бизнеса", baseline)

In [45]:
print(state_3["output"])
print(state_3['reasoning_summary'])

{"score": 0.5, "reasoning": "Сценарий частично соответствует целям социально-экономического развития. Развитие общественного транспорта не поддерживается сценарием (0.0), что является основной причиной низкой оценки. Увеличение зеленых зон также не происходит, так как land_use в сценарии равен LandUse.RECREATION, что указывает на преобладание рекреационных территорий. Вместе с тем, поддержка малого бизнеса может быть частично обеспечена, учитывая presence of business zone (0.0). Оцениваю соответствие в 0.5."}
{"score": 0.5, "reasoning": "Сценарий частично соответствует целям социально-экономического развития. Развитие общественного транспорта не поддерживается сценарием (0.0), что является основной причиной низкой оценки. Увеличение зеленых зон также не происходит, так как land_use в сценарии равен LandUse.RECREATION, что указывает на преобладание рекреационных территорий. Вместе с тем, поддержка малого бизнеса может быть частично обеспечена, учитывая presence of business zone (0.0). О